In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY, OUR_DATASET_DIRECTORY_2, OUR_DATASET_DIRECTORY_3
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=13, micro=12, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [4]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model.load("models_1.6_tests", versionID=3, device=torch.device("cuda:0"))
model.show_infos()

loading the tokenizer from: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/models_save/models_1.6_tests/tokenizer.json
loading the historique from: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/models_save/models_1.6_tests/versions/version_0003_checkpoint-1/history.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(384/768) = 1.414214
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=3, n_kv_head=3, n_embd=384, window_pattern='SSSL')
11_845_932 total params (embeding: 983_040 | last layer: 245_760 | transformer: 10_617_120)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [5]:
print(f"trained for {model.nb_epoches_done} epoches")


trained for 1 epoches


In [7]:
try:
    if "dataset" in globals() : raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)
except FileExistsError: pass

In [8]:
N = 13
print(f"using: {dataset.samples[N].svg_file}")
start = dataset.samples[N].txt[: model.context_size]
#print(start); break
statsPtr: Pointer[GenerationStats] = Pointer()
results = []
for txt in model.generate_flow(
        start=start, decode_batch=256, temperature=1.0, top_k=5, 
        max_tokens=320000, max_time=2*60, statsPtr=statsPtr):
    results.append(txt)
    print(txt, end="", flush=True)

using: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/dataset/samples_100/0013_circle_packing.svg


W0316 11:04:13.092000 448002 .venv/lib/python3.13/site-packages/torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode
W0316 11:04:34.870000 448002 .venv/lib/python3.13/site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] torch._dynamo hit config.recompile_limit (8)
W0316 11:04:34.870000 448002 .venv/lib/python3.13/site-packages/torch/_dynamo/convert_frame.py:1358] [0/8]    function: 'forward' (/autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/LLM/nanochat/gpt.py:388)
W0316 11:04:34.870000 448002 .venv/lib/python3.13/site-packages/torch/_dynamo/convert_frame.py:1358] [0/8]    last reason: 0/7: tensor 'idx' size mismatch at index 1. expected 1690, actual 1691
W0316 11:04:34.870000 448002 .venv/lib/python3.13/site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0316 11:04:34.870000 448002 .venv/lib/python3.13/site-packages/torch/_dynamo/convert_frame.py:1358] [0/8] To diagn

"/><circle r="24.3713" style="fill:none;" cx="56.478" cy="505.7602"/><circle r="26.7698" style="fill:none;" cx="56.478" cy="505.7602"/><circle r="29.1313" style="fill:none;" cx="56.478" cy="505.7602"/><circle r="31.3131" style="fill:none;" cx="569.467" cy="505.2003"/><circle r="33.4221" style="fill:none;" cx="569.467" cy="505.2002"/><circle r="28.3131" style="fill:none;" cx="569.467" cy="505.2002"/><circle r="24.769" style="fill:none;" cx="569.467" cy="505.2002"/><circle r="19.5215" style="fill:none;" cx="569.467" cy="505.2002"/><circle r="17.7676" style="fill:none;" cx="569.467" cy="505.2002"/><circle r="10.1398" style="fill:none;" cx="569.467" cy="505.2002"/><line y2="508.7618" style="fill:none;" x1="55" x2="55.1826" y1="505.7602"/><line y2="508.1931" style="fill:none;" x1="55.1826" x2="84.3126" y1="505.7602"/><line y2="508.2002" style="fill:none;" x1="55.1826" x2="55.1826" y1="505.7602"/><line y2="508.7618" style="fill:none;" x1="55.1826" x2="55.1826" y1="508.2002"/><circle r="5.311

In [9]:
print(start)

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;"><circle r="19.292" style="fill:none;" cx="553.9632" cy="125.0636"/><circle r="11.8105" style="fill:none;" cx="553.9632" cy="125.0636"/><circle r="4.3291" style="fill:none;" cx="553.9633" cy="125.0636"/><circle r="50.677" style="fill:none;" cx="784.2339" cy="110.0319"/><circle r="3.8008" style="fill:none;" cx="784.233

In [10]:
results.insert(0, start)
print(statsPtr.value)
print(f" -> {statsPtr.value.nb_tokens / statsPtr.value.gen_time:.2f} tokens/sec")

GenerationStats(nb_tokens=8249, gen_time=120.00529723, stop_reason='reached max_time')
 -> 68.74 tokens/sec


In [11]:
print(results)


['<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:\'Dialog\'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;"><circle r="19.292" style="fill:none;" cx="553.9632" cy="125.0636"/><circle r="11.8105" style="fill:none;" cx="553.9632" cy="125.0636"/><circle r="4.3291" style="fill:none;" cx="553.9633" cy="125.0636"/><circle r="50.677" style="fill:none;" cx="784.2339" cy="110.0319"/><circle r="3.8008" style="fill:none;" cx="784